In [12]:
import requests
from urllib.parse import urlencode

import pandas as pd
from tqdm import tqdm
import json

BASE_API_URL = 'https://api.gotriple.eu/api/'

In [24]:
params = {
    'q': '',
    'fq': {
        'type': 'typ_article',
        # 'has_pdf': 'true',
        # 'in_language': 'en'
    },
    'include_duplicates': 'false',
    'aggs': {},
    'sort': 'name:desc',
    'page': 1,
    'size': 100, # max 100
    }

In [13]:
def build_api_url(params):
    cleaned_params = {}
    for key, value in params.items():
        if value:
            if isinstance(value, dict):
                if key == 'fq':
                    cleaned_params[key] = urlencode(value).replace('&', ';')
                elif key == 'aggs':
                    cleaned_params[key] = ';'.join([k + ','+ v for k,v in value.items()])
            else:
               cleaned_params[key] = value
    return urlencode(cleaned_params)

In [ ]:
def get_response(query, params):
    params['q'] = query
    endpoint = 'documents'
    objects_list = []
    url = f'{BASE_API_URL}{endpoint}?{build_api_url(params)}'
    response = requests.get(url)
    print('Response length: ', response.json().get('meta').get('total'))
    while True:
        response = requests.get(url)
        if not response.ok:
            print('Response error:', response.status_code)
            print(response.text)
            break
        if not response.json()['data']:
            print('Empty "data" field. End of process.')
            break
        else:
            objects_list.extend(response.json()['data'])
            print(url)
            params['page'] += 1
            if params['page'] == 100:
                print('100 pages reached. End of process.')
                break
            url = f'{BASE_API_URL}{endpoint}?{build_api_url(params)}'

    print(len(objects_list))
    return objects_list

In [15]:
queries = [
    # Society, Culture, and Identity
    '"gender studies" OR "queer theory" OR "sexuality" OR "intersectionality"',
    '"decolonization" OR "postcolonial studies" OR "race and ethnicity" OR "decolonial theory"',
    '"migration" OR "diaspora" OR "refugees" OR "transnationalism"',
    '"cultural heritage" OR "collective memory" OR "heritage restitution" OR "memory politics"',

    # Global Challenges and Social Change
    '"climate change" OR "sustainability" OR "climate justice" OR "environmental humanities"',
    '"inequality" OR "social justice" OR "poverty" OR "social policy"',
    '"urbanization" OR "smart cities" OR "urban justice" OR "housing policy"',
    '"democracy" OR "global governance" OR "authoritarianism" OR "human rights"',

    # Digital Transformation and Technology
    '"digital humanities" OR "text mining" OR "data visualization" OR "digital archives"',
    '"artificial intelligence" OR "AI ethics" OR "algorithmic bias" OR "human-machine interaction"',
    '"social media" OR "misinformation" OR "digital communication" OR "public sphere"',
    '"cyberculture" OR "virtual communities" OR "online identity" OR "digital creativity"',

    # Economy, Work, and Policy
    '"political economy" OR "globalization" OR "neoliberalism" OR "financial crisis"',
    '"automation" OR "future of work" OR "platform economy" OR "gig work"',
    '"public policy" OR "governance" OR "institutional trust" OR "behavioral economics"',
    '"digital education" OR "critical pedagogy" OR "educational equity"',

    # History, Philosophy, and Thought
    '"global history" OR "transnational history" OR "empire" OR "knowledge exchange"',
    '"philosophy of technology" OR "AI ethics" OR "posthumanism" OR "moral responsibility"',
    '"critical theory" OR "power and discourse" OR "knowledge politics" OR "epistemology"',
    '"history of science" OR "biopolitics" OR "pandemics" OR "medical humanities"',

    # Interdisciplinary and Emerging Areas
    '"health humanities" OR "medicine and literature" OR "bioethics"',
    '"environmental sociology" OR "human-nature relations" OR "resilience"',
    '"posthumanism" OR "anthropocene" OR "planetary ethics"',
    '"indigenous knowledge" OR "traditional ecological knowledge" OR "epistemic justice"'
]

In [ ]:
for idx, query in tqdm(enumerate(queries)):
    records = get_response(query, params)
    with open(f'gotriple_response_{idx}.json', 'w', encoding='utf-8') as txt:
        json.dump(records, txt, indent=4, ensure_ascii=False)